# 4 · Exploring ct-domains and ct-SVGs

Plots the tissue domains and the RCC ct-domains, lists the RCC ct-SVGs with gene symbols, and shows the
top ct-SVGs of each RCC domain in space. If notebook 03's optional evaluation runs exist, it also reports
run-to-run ct-SVG stability.

Kernel: `HyperSpot`.

In [ ]:
import pandas as pd
import scanpy as sc

from hyperspot.metadata import ProjectMetadata
from hyperspot.utils import load_ct_spatial_data, load_aggr_st_data, load_decomposed_st_data

from pathlib import Path

# All tutorial outputs live next to these notebooks; run the notebooks from tutorial/.
DATA_DIR      = Path("../demo/data_repo/Human Kidney Cancer")     # raw inputs (see README)
PATIENT_ID    = "PD47171"
PROJECT       = "Human Kidney Cancer Core"
SPOTIPHY_DIR  = Path("spotiphy_project_files") / PROJECT          # Spotiphy inputs/outputs
HYPERSPOT_DIR = Path("hyperspot_project_files") / PROJECT         # HyperSpot project (the CLI resolves
                                                                  # --project relative to ./hyperspot_project_files)
ProjectMetadata.use(HYPERSPOT_DIR)
pm = ProjectMetadata()
SYMBOL = load_decomposed_st_data().var["SYMBOL"].astype(str)

## Tissue domains and ct-domains

In [ ]:
from hyperspot.visualization import plot_aggr_st_domains, plot_all_ct_domains
plot_aggr_st_domains(display=True, save=False)
plot_all_ct_domains()

## RCC ct-SVGs

In [ ]:
CT = "RCC"
ct_svgs = pd.read_csv(HYPERSPOT_DIR / "svgs" / f"svgs_ct_{CT}.csv", index_col=0)
ct_svgs.insert(0, "symbol", SYMBOL.reindex(ct_svgs.index).values)
print(f"{len(ct_svgs)} {CT} ct-SVGs across {ct_svgs['region'].nunique()} domains")
ct_svgs.sort_values("scores", ascending=False).head(20)

In [ ]:
ct_svgs.groupby("region").size().rename("ct-SVGs").to_frame().T

## Top ct-SVGs of each RCC domain, in space

For each domain, the highest-scoring ct-SVGs are shown on the RCC cells. Each panel highlights where the
gene is enriched.

In [ ]:
from hyperspot.pipeline import compute_domain_ct_svgs
from hyperspot.visualization.svg import plot_svgs_by_gene

adata_ct = load_ct_spatial_data(CT)
domain_svgs = compute_domain_ct_svgs(CT, min_pts=0.8, min_logfc=1.5)       # {domain: DataFrame}
plot_svgs_by_gene(adata_ct, domain_svgs, per="top", limit=4, save=False)

## Run-to-run stability (optional)

Needs the `perform-model-evaluation` runs from notebook 03. Reports, per cell type, the mean pairwise
containment |A∩B| / min(|A|,|B|) between the 10 seed runs' ct-SVG sets.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from conserved_ctsvg import run_stability

if (HYPERSPOT_DIR / "evaluation").exists():
    display(run_stability(str(HYPERSPOT_DIR), pm.ct_list).round(3))
else:
    print("No evaluation runs yet; run perform-model-evaluation in notebook 03.")